# 🎬 Module 12 · Opening Demo: Watch an AI Agent Think
**Designed by Engr Usman Hasan** · Bano Qabil Generative AI course

**Teacher's opening for Saturday.** Run 3 cells, project the screen, and let the class watch.

| Cell | What the class sees |
|---|---|
| ⚙️ Setup | nothing (run it before class) |
| 🧠 Engine | "Agent engine ready" (run it before class) |
| 🎬 The show | a plain chatbot **fails**, then the agent **loops**: Think → Act → Observe → … → Answer |
| 🔁 More | a different question = a different number of loops |

**Key:** Colab: 🔑 Secrets → `GROQ_API_KEY` (Notebook access ON). VS Code: put it in `.env`. Free key: https://console.groq.com/keys

In [ ]:
# ===== SETUP: run this once (works in Colab AND VS Code) =====
# install the Groq and Gemini packages quietly (python-dotenv reads a .env file in VS Code)
%pip install -q "groq==1.7.0" "google-genai==3.6.1" "python-dotenv==1.2.4"

# 'os' holds the API keys
import os

# choose the brain: "groq" or "gemini"
PROVIDER = "groq"
# the key's name for that provider
KEY_NAME = "GROQ_API_KEY" if PROVIDER == "groq" else "GEMINI_API_KEY"

# try Colab first: keys live in the 🔑 Secrets panel
try:
    # Colab's secrets reader
    from google.colab import userdata
    # copy the secret into the environment
    os.environ[KEY_NAME] = userdata.get(KEY_NAME)
# not in Colab (e.g. VS Code): read the .env file instead
except Exception:
    # python-dotenv reads .env
    from dotenv import load_dotenv
    # load the keys
    load_dotenv()

# confirm the key is there (we never print the key itself!)
print(f"✅ {KEY_NAME} ready" if os.getenv(KEY_NAME) else f"⚠️ {KEY_NAME} missing: add it to Colab Secrets or .env")

In [ ]:
# ===== THE AGENT ENGINE (just run it; we build all of this step by step in the Day 1 notebook) =====
# 're' reads the LLM's Thought / Action lines
import re
# 'time' adds small pauses so the class can follow each step
import time
# 'html' makes text safe to show inside HTML
import html
# display + HTML draw coloured boxes in the notebook
from IPython.display import display, HTML

# seconds to wait between steps (make it 0 to go fast)
DELAY = 1.5
# the model for each provider (change here if a name is retired)
MODELS = {"groq": "llama-3.3-70b-versatile", "gemini": "gemini-flash-latest"}

# ---------- the brain: send messages to Groq or Gemini, get text back ----------
# Groq client
if PROVIDER == "groq":
    # Groq's package
    from groq import Groq
    # connect with our key
    client = Groq(api_key=os.environ["GROQ_API_KEY"])
# Gemini client
else:
    # Google's package
    from google import genai
    # connect with our key
    client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])


# one LLM call: messages in, text out
def ask_llm(messages, stop=None):
    # Groq
    if PROVIDER == "groq":
        # send the conversation (temperature 0 = no randomness)
        r = client.chat.completions.create(model=MODELS["groq"], messages=messages, temperature=0, stop=stop)
        # the reply text
        return r.choices[0].message.content or ""
    # Gemini: assistant is called "model", and the system message goes in the config
    contents = [{"role": "model" if m["role"] == "assistant" else "user", "parts": [{"text": m["content"]}]} for m in messages[1:]]
    # send it
    r = client.models.generate_content(model=MODELS["gemini"], contents=contents,
                                       # system message + no randomness + stop words
                                       config={"system_instruction": messages[0]["content"], "temperature": 0, "stop_sequences": stop})
    # the reply text
    return r.text or ""


# ---------- the farm's data ----------
# prices in rupees
PRICES = {"milk": 220, "yogurt": 250, "desi ghee": 2600}
# units
UNITS = {"milk": "litre", "yogurt": "kg", "desi ghee": "kg"}
# today's stock
STOCK = {"milk": 40, "yogurt": 12, "desi ghee": 3}
# the farm's rules
RULES = ["Delivery areas: DHA, Clifton, Gulshan and PECHS only.",
         # charge
         "Delivery charge: Rs 150 per order. Free delivery when the bill is Rs 3000 or more.",
         # discount
         "Bulk discount: 5% off milk when a customer orders 20 litres or more.",
         # time
         "Delivery time: every morning from 6 am to 9 am."]


# ---------- the tools (the agent's hands) ----------
# price of a product
def get_price(product):
    """Price of ONE farm product. Input: product name (milk, yogurt or desi ghee)."""
    # clean the input
    p = product.lower().strip()
    # answer
    return f"{p}: Rs {PRICES[p]} per {UNITS[p]}" if p in PRICES else f"We only sell: {', '.join(PRICES)}"


# stock of a product
def check_stock(product):
    """How much of ONE product is in stock today. Input: product name."""
    # clean the input
    p = product.lower().strip()
    # answer
    return f"{p}: {STOCK[p]} {UNITS[p]} in stock" if p in STOCK else f"We only sell: {', '.join(STOCK)}"


# maths
def calculator(expression):
    """Do maths. Input: numbers with + - * / ( ), e.g. 15 * 220."""
    # allow only plain maths (safety)
    if any(c not in "0123456789+-*/(). " for c in expression):
        # refuse anything else
        return "Error: only numbers and + - * / ( )"
    # calculate
    try:
        # the result, rounded
        return str(round(eval(expression), 2))
    # broken maths
    except Exception as e:
        # tell the agent
        return f"Error: {e}"


# search the rules
def search_farm_rules(question):
    """Search the farm's rules: delivery areas, delivery charge, discounts, delivery time. Input: a short question."""
    # the question's longer words
    words = [w.strip("?.,!") for w in question.lower().split() if len(w) > 3]
    # rules that share a word with the question
    found = [r for r in RULES if any(w in r.lower() for w in words)]
    # answer
    return "\n".join(found[:2]) or "No rule found."


# the toolbox: name -> function
TOOLS = {"get_price": get_price, "check_stock": check_stock, "calculator": calculator, "search_farm_rules": search_farm_rules}
# an icon for each tool (just for the show)
ICONS = {"get_price": "🏷️", "check_stock": "📦", "calculator": "🧮", "search_farm_rules": "📖"}

# the ReAct instructions
SYSTEM = """You are the agent of Marhaba Cattle & Dairy Farm in Karachi.
Tools:
""" + "\n".join(f"- {n}: {f.__doc__}" for n, f in TOOLS.items()) + """

Reply in EXACTLY this format, then STOP and wait for the Observation:
Thought: what to do next
Action: one tool name
Action Input: the input

When you know the answer:
Thought: I now know the final answer
Final Answer: a short, friendly answer in the customer's language

Never guess prices, stock, rules or maths: use a tool. One tool at a time."""


# ---------- pretty boxes ----------
# draw one coloured box
def box(icon, label, text, color, bg):
    # an HTML card with a coloured left label
    display(HTML(f"""<div style="font-family:Segoe UI,Arial;margin:6px 0 6px 40px;padding:10px 14px;border-radius:10px;background:{bg};border:2px solid {color};max-width:820px">
      <b style="color:{color};font-size:15px">{icon} {label}</b><div style="font-size:16px;color:#1F2D2B;margin-top:4px;white-space:pre-wrap">{html.escape(text)}</div></div>"""))
    # pause so everyone can read it
    time.sleep(DELAY)


# draw a big banner
def banner(text, color="#113A37"):
    # dark wide box with white text
    display(HTML(f"""<div style="font-family:Segoe UI,Arial;margin:14px 0 6px 0;padding:10px 16px;border-radius:10px;background:{color};color:white;font-size:18px;font-weight:bold;max-width:860px">{text}</div>"""))


# ---------- 1) a plain chatbot (no tools, no loop) ----------
def chatbot(question):
    # show the question
    banner("🤖 PLAIN CHATBOT: one call, no tools, no loop", "#E8575A")
    # the customer
    box("🙋", "CUSTOMER", question, "#2F6FDB", "#E6EFFC")
    # one LLM call with no data
    answer = ask_llm([{"role": "system", "content": "You are the assistant of Marhaba Cattle & Dairy Farm in Karachi. Answer briefly."},
                      # the question
                      {"role": "user", "content": question}])
    # show the answer
    box("💬", "CHATBOT ANSWER (1 LLM call, nothing checked)", answer.strip(), "#E8575A", "#FDE8E8")


# ---------- 2) the agent: Think -> Act -> Observe -> repeat ----------
def agent(question, max_steps=8):
    # title
    banner("🦾 AI AGENT: LLM + tools + a loop")
    # the customer
    box("🙋", "CUSTOMER", question, "#2F6FDB", "#E6EFFC")
    # the conversation starts with instructions + the question
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    # tools used, for the summary
    used = []
    # the loop (with a seatbelt)
    for loop in range(1, max_steps + 1):
        # loop number
        banner(f"🔁 LOOP {loop}", "#7C5CDB")
        # THINK: ask the LLM what to do next (stop before it invents an Observation)
        reply = ask_llm(messages, stop=["Observation:"]).strip()
        # the Thought line (if any)
        thought = re.search(r"Thought:\s*(.+)", reply)
        # show the thinking
        box("🧠", "THINK", thought.group(1).strip() if thought else reply, "#7C5CDB", "#EEE9FB")
        # finished?
        if "Final Answer:" in reply:
            # the answer text
            final = reply.split("Final Answer:")[-1].strip()
            # show it
            box("✅", "FINAL ANSWER", final, "#2F9E6E", "#E3F4EC")
            # the summary
            banner(f"🏁 Done in {loop} loops · {loop} LLM calls · tools used: {' → '.join(used) or 'none'}", "#2F9E6E")
            # stop the loop
            return final
        # which tool and which input?
        action = re.search(r"Action:\s*(.+)", reply)
        # the input line
        action_input = re.search(r"Action Input:\s*(.+)", reply)
        # wrong format -> remind it and loop again
        if not (action and action_input):
            # the reminder
            observation = "Wrong format. Use Thought / Action / Action Input, or Final Answer."
        # a real tool
        else:
            # the tool name
            tool = action.group(1).strip()
            # its input
            tool_input = action_input.group(1).strip().strip('"')
            # show the action
            box(ICONS.get(tool, "🔧"), "ACT (our Python code runs the tool)", f"{tool}({tool_input})", "#2F6FDB", "#E6EFFC")
            # run it (or explain it doesn't exist)
            observation = TOOLS[tool](tool_input) if tool in TOOLS else f"No tool called {tool}. Use: {', '.join(TOOLS)}"
            # remember it for the summary
            used.append(tool)
        # OBSERVE: show the result
        box("👀", "OBSERVE (the agent reads the result)", observation, "#1E8C88", "#E4F4F3")
        # add the reply and the result to the conversation for the next loop
        messages += [{"role": "assistant", "content": reply}, {"role": "user", "content": f"Observation: {observation}"}]
    # the seatbelt stopped it
    banner(f"⛔ Stopped after {max_steps} loops (max_steps seatbelt)", "#E8575A")


# ready
print("✅ Agent engine ready. Tools:", ", ".join(TOOLS))

## 🎬 The show
Ask the class first: **"Can a chatbot tell us today's price, check the stock AND do the maths?"** Then run the cell.

In [ ]:
# ===== 🎬 THE SHOW: same question, chatbot vs agent =====
# a real customer message that needs price + stock + maths
QUESTION = "Ali ko kal 15 litre doodh chahiye. Total bill kitna banega, aur kya itna stock hai?"

# 1) the plain chatbot tries
chatbot(QUESTION)
# 2) the agent tries: watch it LOOP
agent(QUESTION)

### Say this to the class
- The chatbot made **1 call** and **guessed** (or gave up). It has no hands.
- The agent **looped**: each loop it **thought**, **used a tool**, **read the result**, and decided the next step **by itself**.
- **Agent = LLM (brain) + tools (hands) + a loop.** Today, you build this yourself, from scratch.

*Roman Urdu: Chatbot sirf jawab deta hai. Agent kaam karta hai: socho, karo, dekho, phir socho.*

In [ ]:
# ===== 🔁 MORE QUESTIONS: watch the number of loops change =====
# pick one by removing the # in front of it (or write your own)
QUESTION = "I live in Clifton. 25 litre doodh discount aur delivery ke saath kitne ka padega?"
# QUESTION = "Desi ghee kitne ka hai?"
# QUESTION = "Assalam o alaikum! Aap kaun ho?"
# QUESTION = "Kya aap Malir mein delivery karte ho?"

# run the agent on it
agent(QUESTION)

**Ask the class:** "Why did the greeting need 1 loop but the Clifton question needed 4 or 5?" → *The agent decides how many steps the job needs. Nobody wrote those steps for it.*

---
© Module 12 course material designed by **Engr Usman Hasan**.